In [1]:
import geemap
import ee
ee.Initialize(project='pyregence-ee')
#import scripts.analysis_functions as af
#import scripts.fire_info as fi
#import scripts.get_image_collections as gic
import scripts.utils as utils
from generate_model_inputs import get_gridmet, get_mtbs_bs, bi_calc, construct_stack

## Construct Stack of Predictors and Response Class Layer

### Response Class - MTBS Burn Severity (Stripe Corrected)

### Precictors:
#### * Landsat dNBR
#### * Landsat RdNBR
#### * GridMet erc (energ_release)
#### * GridMet tmmx (t_max)
#### * GridMet tmmn (t_min)
#### * GridMet th (wind_direction)
#### * GridMet vs (wind_velocity)
#### * GridMet bi (burn ind)
#### * GridMet fm100 (100hr fuel moisture)
#### * GridMet fm1000 (1000hr fuel moisture)
#### * LANDFIRE EVT (Existing Vegetation Type)

In [8]:
# Bring in MTBS fires processed
fires = ee.FeatureCollection("projects/pyregence-ee/assets/conus/mtbs/mtbs_perims_processed")
# remove HI and AK fires
conus_bounds = ee.Image("projects/pyregence-ee/assets/conus/landfire/zones_image").geometry()
fires = fires.filterBounds(conus_bounds) 
#print(fires.size().getInfo())

no_scene_id = ee.Filter.neq('Pre_ID', '') # remove fires that don't have a Pre_ID scene string, need this for Landsat compositing
not_sentinel = (ee.Filter.stringContains('Pre_ID', '30m')).Not() # remove fires whose pre scene id was a Sentinel one, only doing Landsat for now
#year_filter = ee.Filter.Or( ee.Filter.eq('fire_yr', 2017), ee.Filter.eq('fire_yr', 2012), ee.Filter.eq('fire_yr', 1996) ) 

fires_subset = fires.filter(ee.Filter.And(no_scene_id, not_sentinel))
fires_smaller = fires_subset.limit(5, 'BurnBndAc', False)
#print(fires_smaller.size().getInfo())
#print(fires_subset.size().getInfo())
fires_smaller_simple = fires_smaller.map(lambda feat: feat.simplify(1))
#test individual data getters 
# fires_bi_imgs = ee.ImageCollection(fires_subset.map(bi_calc)).mosaic()
# print(fires_bi_imgs.bandNames().getInfo())
# gridmet_imgs = ee.ImageCollection(fires_subset.map(get_gridmet)).mosaic()
# print(gridmet_imgs.bandNames().getInfo())
# mtbs_imgs = ee.ImageCollection(fires_subset.map(get_mtbs_bs)).mosaic()
# print(mtbs_imgs.bandNames().getInfo())

# using the stack constructer which runs all of the above functions and stacks the images together
stack = construct_stack(fires_smaller_simple)
input_bands = stack.bandNames().remove('SEVERITY')


In [9]:
Map = geemap.Map()

#Map.centerObject(ee.Feature(fires_subset.first()), 8)


# Map.addLayer(mtbs_imgs, {'min':0, 'max':6, 'palette': ["00ff1f","fbff0e","ffbc00","ff0000", 'red', 'grey']}, 'mtbs')
Map.addLayer(stack, {}, 'stack', False)
Map.addLayer(fires_smaller, {'color':'pink'}, 'fires subset', True)
Map.addLayer(fires_smaller_simple, {'color': 'purple'}, 'fires subset simplified', True, 0.7)

Map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(T…

In [11]:
# Create random sampling point (test and train points)
#Map = geemap.Map()
trainPoints = ee.FeatureCollection.randomPoints(region= fires_smaller_simple, points= 7000, seed= 0, maxError= 1)
#Map.addLayer(trainPoints, {'color': 'black'}, 'Train points', False)

testPoints = ee.FeatureCollection.randomPoints(region= fires_smaller_simple, points= 3000, seed= 0, maxError= 1) # tried using a version of fires that were .simplify() 'ed
#Map.addLayer(testPoints, {'color': 'black'}, 'Test points', False)

# Overlay training points on corresponding composite image to create FeatureCollection
training = stack.sampleRegions(collection=trainPoints, scale=30, geometries=False, tileScale=1);  #tileScale=1 to reduce the chances of running into memory problem, omit geometries to save memory and avoid GEE timing out
#print(training.first().getInfo()) 

testing = stack.sampleRegions(collection=testPoints, scale=30, geometries=False, tileScale=1)
#Map.centerObject(trainPoints.first(), 14)
#Map

In [12]:
# ========== Classification MODEL
def fitmodel_class(responsevar_class):
  rf = ee.Classifier.smileRandomForest(
                    numberOfTrees= 500,
                    #variablesPerSplit=6,
                    #minLeafPopulation= 5,
                    #bagFraction= 0.5,
                    #outOfBagMode= True,
                  ).setOutputMode('CLASSIFICATION')

  trained_classifier = rf.train(features= training,classProperty= responsevar_class,inputProperties= input_bands)

  dict = trained_classifier.explain()
  #print('Explain', responsevar, dict)

  classified = stack.classify(trained_classifier)

  #print ('classified', classified)

  # ===== map prediction and original burn severity

 
  
  #Map.addLayer(orginal_bs,{}, 'MTBS Burn Severity')
  Map.addLayer(classified,{}, 'Classified Severity')

  # ====== QAQA
  # == Train data confusion matrix
  trainAccuracy = trained_classifier.confusionMatrix()
  print('Resubstitution error matrix: ', trainAccuracy)
  print('Training overall accuracy: ', trainAccuracy.accuracy())

  # ====== Test data
  validated = testing.classify(trained_classifier)
  testAccuracy = validated.errorMatrix('new_fbfm40', 'classification')
  print('Validation error matrix: ', testAccuracy)
  print('Validation overall accuracy: ', testAccuracy.accuracy())

fitmodel_class('SEVERITY')
Map


EEException: User memory limit exceeded.

In [ ]:
# Map = geemap.Map()

# Map.addLayer(bs_finalv1, {"min":0,"max":3,"palette":["00ff1f","fbff0e","ffbc00","ff0000"]},'bs v1')
# Map.addLayer(bs_finalv2, {"min":0,"max":3,"palette":["00ff1f","fbff0e","ffbc00","ff0000"]},'bs v2')

# #Map.addLayer(featCollv1, {'color':'pink'}, 'fires', True,0.4)
# Map.setCenter(-112, 39, 6)

# Map

In [ ]:
# export img to asset
# desc = 'bs_2020fires_gte500ac_v1'
# utils.exportImgtoAsset(bs_finalv1, desc=desc, export=False)

# desc = 'bs_2020fires_gte500ac_v2'
# utils.exportImgtoAsset(bs_finalv2, desc=desc, export=False)